In [2]:
import os
os.environ["JAX_PLATFORM_NAME"] = "gpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import jax
print(jax.devices())
print("default backend:", jax.default_backend())


[CudaDevice(id=0)]
default backend: gpu


In [4]:
# Flags to improve determinism.
os.environ['XLA_FLAGS'] = ' '.join([
    '--xla_gpu_deterministic_ops',
    '--xla_gpu_enable_scatter_determinism_expander=True',
    '--xla_gpu_enable_triton_gemm=False',
])
# Increase GPU and CPU memory to reduce out of memory errors.
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9'

In [5]:
# @title Imports

import functools
import os
from typing import Callable

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn import metrics
from sklearn.metrics import average_precision_score, roc_auc_score

from alphagenome.data import genome
from alphagenome.visualization import plot_components
from alphagenome_research.model import dna_model
import matplotlib.pyplot as plt
from alphagenome.models import variant_scorers
from alphagenome.models import dna_client

I0000 00:00:1776782552.798641 1862590 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776782555.462611 1862590 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [6]:
# @title Helper functions.

def calculate_tissue_weighted_metric(
    predictions_df: pd.DataFrame,
    metric_fn: Callable[[np.ndarray, np.ndarray], float],
    prediction_col: str = 'prediction',
    target_col: str = 'target',
    tissue_col: str = 'tissue'
) -> float:
    """
    Calculates a tissue-weighted mean for a given metric function from a DataFrame.

    Args:
        predictions_df: DataFrame containing predictions, targets, and tissue info.
        metric_fn: A function that accepts two np.ndarrays (y_true, y_pred)
                   and returns a single float metric.
                   Note: For scipy functions that return tuples (e.g., spearmanr),
                   you must wrap them in a lambda function (see example).
        prediction_col: Name of the column with prediction scores.
        target_col: Name of the column with target labels/values.
        tissue_col: Name of the column indicating the tissue/group.

    Returns:
        The calculated tissue-weighted mean metric, or np.nan if unable to calculate.
    """
    if predictions_df is None or predictions_df.empty:
        print("Warning: predictions_df is empty, returning NaN.")
        return float('nan')

    required_cols = [prediction_col, target_col, tissue_col]
    if not all(col in predictions_df.columns for col in required_cols):
        raise ValueError(f"DataFrame must contain columns: {required_cols}")

    tissue_results = []

    # Group by tissue and calculate the metric for each one.
    for tissue_name, group_df in predictions_df.groupby(tissue_col):

        # Drop rows where target or prediction is NaN.
        clean_group = group_df.dropna(subset=[target_col, prediction_col])

        num_variants = len(clean_group)

        # Skip tissues with < 2 variants (can't calculate metrics).
        if num_variants < 2:
            # print(f"Skipping tissue {tissue_name} (variants={num_variants})")
            continue

        try:
            # Calculate the metric for this one tissue using the passed function
            metric_value = metric_fn(
                clean_group[target_col], clean_group[prediction_col]
            )

            # Ensure metric is a valid number.
            if not np.isfinite(metric_value):
                print(f"Skipping tissue {tissue_name} (metric_fn returned non-finite value)")
                continue

            tissue_results.append({
                'tissue': tissue_name,
                'metric_value': metric_value,
                'count': num_variants  # This is the "weight".
            })
        except ValueError as e:
            # This will catch single-class errors for AUROC/AUPRC
            # and other potential issues from metric_fn.
            print(f"Could not calculate metric for tissue {tissue_name}: {e}")
            continue # Skip tissue if metric calculation fails.

    if not tissue_results:
        print("Warning: No tissues had scorable metric values, returning NaN.")
        return float('nan')

    # Create a DataFrame of the per-tissue metrics.
    metrics_df = pd.DataFrame(tissue_results)

    # Calculate the final weighted mean.
    weighted_sum = (metrics_df['metric_value'] * metrics_df['count']).sum()
    total_count = metrics_df['count'].sum()

    if total_count == 0:
         print("Warning: Total count for weighting is zero, returning NaN.")
         return float('nan')

    weighted_mean_metric = weighted_sum / total_count
    return weighted_mean_metric

# qtl
def paqtl_auprc(df):
  SEED = 0

  # 1. Separate positives and negatives.
  pos = df[df['target'] == 1]
  neg = df[df['target'] == 0]

  # 2. Merge on 'PI' to find all valid matched pairs.
  matched = pd.merge(
      pos,
      neg,
      on='PI',
      how='inner',
      suffixes=('_pos', '_neg')
  )

  # 3. Group by 'PI' and sample exactly one pair per group.
  # This ensures 1:1 matching controlled by PI.
  sampled_pairs = matched.groupby('PI').sample(n=1, random_state=SEED)

  # 4. Reconstruct a single dataframe for AUPRC calculation
  # We stack the positive and negative parts of the sampled pairs back together.
  df_sampled = pd.concat([
      sampled_pairs[['prediction_pos', 'target_pos']].rename(
          columns={'prediction_pos': 'prediction', 'target_pos': 'target'}
      ),
      sampled_pairs[['prediction_neg', 'target_neg']].rename(
          columns={'prediction_neg': 'prediction', 'target_neg': 'target'}
      )
  ])
  return metrics.average_precision_score(
      df_sampled['target'], df_sampled['prediction'])

auroc_fn = roc_auc_score
auprc_fn = average_precision_score

spearman_fn = lambda y_true, y_pred: spearmanr(y_true, y_pred)[0]


In [7]:
# @title Eval configs.
PREDS_PATH = 'https://storage.googleapis.com/alphagenome/evals'

evals = {
    'clinvar_splice_site_region': {
        'output_type':      'SPLICE_SITE_USAGE;RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_JUNCTIONS;SPLICE_SITES',
        'metric_name':      'auprc_max_abs_track_aggregation',
        'metric_fn':        lambda x: metrics.average_precision_score(x['target'], x['prediction']),
        'reported_metric':  0.5699,
    },
    'clinvar_noncoding': {
        'output_type':      'RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAGE;SPLICE_JUNCTIONS;SPLICE_SITES',
        'metric_name':      'auprc_max_abs_track_aggregation',
        'metric_fn':        lambda x: metrics.average_precision_score(x['target'], x['prediction']),
        'reported_metric':  0.6588,
    },
    'clinvar_missense': {
        'output_type':      'SPLICE_SITE_USAGE;SPLICE_JUNCTIONS;SPLICE_SITES;RNA_SEQ;SPLICE_SITE_POSITIONS',
        'metric_name':      'auprc_max_abs_track_aggregation',
        'metric_fn':        lambda x: metrics.average_precision_score(x['target'], x['prediction']),
        'reported_metric':  0.1792,
    },
    'sqtl_variant_causality_gene_human': {
        'output_type':      'RNA_SEQ;SPLICE_JUNCTIONS;SPLICE_SITES;SPLICE_SITE_USAGE;SPLICE_SITE_POSITIONS',
        'metric_name':      'tissue_weighted_mean_auprc',
        'metric_fn':        functools.partial(calculate_tissue_weighted_metric, metric_fn=auprc_fn),
        'reported_metric':  0.7644,
    },
    'mfass_splicing': {
        'output_type':      'SPLICE_SITES_LOGITS;SPLICE_SITE_USAGE;SPLICE_JUNCTIONS',
        'metric_name':      'all_tissues_auprc',
        'metric_fn':        lambda x: metrics.average_precision_score(x['target'], x['prediction']),
        'reported_metric':  0.5120,
    },
    'eqtl_variant_borzoi_sign_human': {
        'output_type':      'RNA_SEQ',
        'metric_name':      'tissue_weighted_mean_auroc',
        'metric_fn':        functools.partial(calculate_tissue_weighted_metric, metric_fn=auroc_fn),
        'reported_metric':  0.810077,
    },
    'eqtl_variant_catalogue_causality_gene_balanced_human': {
        'output_type':      'RNA_SEQ',
        'metric_name':      'tissue_weighted_mean_auroc',
        'metric_fn':        functools.partial(calculate_tissue_weighted_metric, metric_fn=auroc_fn),
        'reported_metric':  0.713255,
    },
    'eqtl_variant_borzoi_coefficient_human': {
        'output_type':      'RNA_SEQ',
        'metric_name':      'tissue_weighted_mean_spearmanr',
        'metric_fn':        functools.partial(calculate_tissue_weighted_metric, metric_fn=spearman_fn),
        'reported_metric':  0.500588,
    },
    'enhancer_gene_linking_e2g': {
        'output_type':      'RNA_SEQ',
        'metric_name':      None,
        'metric_fn':        lambda x: metrics.average_precision_score(x['target'], x['prediction']),
        'reported_metric':  0.7490,
    },
    'paqtl_variant_causality_human': {
        'output_type':      'RNA_SEQ',
        'metric_name':      'PAS_10000_threshold_average_auprc',
        'metric_fn':        paqtl_auprc,
        'reported_metric':  0.6294,
        'notes':            'The reported metric is from many permutations, here we just do 1 resample.',
    },
    'caqtl_african_variant_causality_human': {
        'output_type':      'DNASE',
        'metric_name':      "auprc_mean_abs_track_aggregation_ontology_curie:['EFO:0002784']",
        'metric_fn':        lambda x: metrics.average_precision_score(x['target'], x['prediction']),
        'reported_metric':  0.5643,
    },
    'caqtl_european_variant_causality_human': {
        'output_type':      'DNASE',
        'metric_name':      "auprc_mean_abs_track_aggregation_ontology_curie:['EFO:0002784']",
        'metric_fn':        lambda x: metrics.average_precision_score(x['target'], x['prediction']),
        'reported_metric':  0.3638,
    },
    'dsqtl_yoruba_variant_causality_human': {
        'output_type':      'DNASE',
        'metric_name':      "auprc_mean_abs_track_aggregation_ontology_curie:['EFO:0002784']",
        'metric_fn':        lambda x: metrics.average_precision_score(x['target'], x['prediction']),
        'reported_metric':  0.6308,
    },
    'caqtl_african_variant_coefficient_human': {
        'output_type':      'DNASE',
        'metric_name':      "pearsonr_mean_track_aggregation_ontology_curie:['EFO:0002784']",
        'metric_fn':        lambda x: pearsonr(x['target'], x['prediction']).statistic,
        'reported_metric':  0.7368,
    },
    'caqtl_european_variant_coefficient_human': {
        'output_type':      'DNASE',
        'metric_name':      "pearsonr_mean_track_aggregation_ontology_curie:['EFO:0002784']",
        'metric_fn':        lambda x: pearsonr(x['target'], x['prediction']).statistic,
        'reported_metric':  0.5916,
    },
    'dsqtl_yoruba_variant_coefficient_human': {
        'output_type':      'DNASE',
        'metric_name':      "pearsonr_mean_track_aggregation_ontology_curie:['EFO:0002784']",
        'metric_fn':        lambda x: pearsonr(x['target'], x['prediction']).statistic,
        'reported_metric':  0.8323,
        'notes':            'The sign of the correlation flips depending on which allele is labelled REF vs. ALT.',
    },
    'caqtl_microglia_variant_coefficient_human': {
        'output_type':      'DNASE',
        'metric_name':      "pearsonr_mean_track_aggregation_ontology_curie:['CL:0000862']",
        'metric_fn':        lambda x: pearsonr(x['target'], x['prediction']).statistic,
        'reported_metric':  0.6357,
        'notes':            'The sign of the correlation flips depending on which allele is labelled REF vs. ALT.',
    },
    'caqtl_smc_variant_coefficient_human': {
        'output_type':      'ATAC',
        'metric_name':      "pearsonr_mean_track_aggregation_ontology_curie:['UBERON:0002079']",
        'metric_fn':        lambda x: pearsonr(x['target'], x['prediction']).statistic,
        'reported_metric':  0.687,
    },
    'bqtl_spi1_variant_coefficient_human': {
        'output_type':      'CHIP_TF',
        'metric_name':      'pearsonr_mean_track_aggregation_EFO:0002784_SPI1',
        'metric_fn':        lambda x: pearsonr(x['target'], x['prediction']).statistic,
        'reported_metric':  0.549967,
    },
    'bqtl_spi1_variant_causality_human': {
        'output_type':      'CHIP_TF',
        'metric_name':      'auprc_mean_abs_track_aggregation_EFO:0002784_SPI1',
        'metric_fn':        lambda x: metrics.average_precision_score(x['target'], x['prediction']),
        'reported_metric':  0.4952,
    },
}

In [8]:
url = "https://storage.googleapis.com/alphagenome/evals/clinvar_noncoding_predictions.feather"
df = pd.read_feather(url)

row = df.iloc[0]
print(row)

variant_id = row["variant_id"]

chrom, pos, ref, alt, build = variant_id.split("_")
pos = int(pos)

variant = genome.Variant(
    chromosome=chrom,
    position=pos,
    reference_bases=ref,
    alternate_bases=alt,
)

def parse_variant_id(variant_id: str) -> genome.Variant:
    chrom, pos, ref, alt, build = variant_id.split("_")
    return genome.Variant(
        chromosome=chrom,
        position=int(pos),
        reference_bases=ref,
        alternate_bases=alt,
    )

df['variant'] = df['variant_id'].apply(parse_variant_id)

interval = variant.reference_interval.resize(2**20)

variant_id                                       chr1_1043197_G_C_hg38
prediction                                                    0.016315
target                                                             1.0
variant_scorer                               ensemble_scorer;sj,ssu,ss
output_type          RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...
metric_calculator                      SimpleAggregatorAUPRCCalculator
metric_name                            auprc_max_abs_track_aggregation
Name: 0, dtype: object


In [9]:
df.output_type[0]

'RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAGE;SPLICE_JUNCTIONS;SPLICE_SITES'

In [10]:
model = dna_model.create_from_huggingface('all_folds')


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/opt/modules/i12g/anaconda/envs/IDPproject_indel_26/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/opt/modules/i12g/anaconda/envs/IDPproject_indel_26/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


In [11]:

urls = {
    'clinvar_noncoding': 'https://storage.googleapis.com/alphagenome/evals/clinvar_noncoding_predictions.feather',
    'clinvar_splice_site_region': 'https://storage.googleapis.com/alphagenome/evals/clinvar_splice_site_region_predictions.feather',
    'clinvar_missense': 'https://storage.googleapis.com/alphagenome/evals/clinvar_missense_predictions.feather',
}

dfs = {}
for name, url in urls.items():
    try:
        dfs[name] = pd.read_feather(url)
        print(f"{name}: {dfs[name].shape}")
    except Exception as e:
        print(f"{name}: FAILED - {e}")

clinvar_noncoding: (96688, 7)
clinvar_splice_site_region: (17955, 7)
clinvar_missense: (35244, 7)


In [12]:
metadata = model.output_metadata()

# Check splice-related output types
print("=== SPLICE_SITES ===")
print(metadata.splice_sites)

print("\n=== SPLICE_SITE_USAGE ===")
print(metadata.splice_site_usage.head(10))
print(f"Total tracks: {len(metadata.splice_site_usage)}")
print(f"Unique ontologies: {metadata.splice_site_usage['ontology_curie'].nunique()}")
print(metadata.splice_site_usage['ontology_curie'].unique())

print("\n=== SPLICE_JUNCTIONS ===")
print(metadata.splice_junctions.head(10))
print(f"Total tracks: {len(metadata.splice_junctions)}")
print(f"Unique ontologies: {metadata.splice_junctions['ontology_curie'].nunique()}")

=== SPLICE_SITES ===
       name strand
0     donor      +
1  acceptor      +
2     donor      -
3  acceptor      -

=== SPLICE_SITE_USAGE ===
                                  name strand         Assay title  \
0  usage_CL:0000047 polyA plus RNA-seq      +  polyA plus RNA-seq   
1       usage_CL:0000062 total RNA-seq      +       total RNA-seq   
2  usage_CL:0000084 polyA plus RNA-seq      +  polyA plus RNA-seq   
3       usage_CL:0000084 total RNA-seq      +       total RNA-seq   
4       usage_CL:0000100 total RNA-seq      +       total RNA-seq   
5       usage_CL:0000115 total RNA-seq      +       total RNA-seq   
6  usage_CL:0000121 polyA plus RNA-seq      +  polyA plus RNA-seq   
7       usage_CL:0000127 total RNA-seq      +       total RNA-seq   
8  usage_CL:0000134 polyA plus RNA-seq      +  polyA plus RNA-seq   
9       usage_CL:0000137 total RNA-seq      +       total RNA-seq   

  ontology_curie         biosample_name                 biosample_type  \
0     CL:0000047     ne

In [13]:
import numpy as np
from alphagenome.data import genome
from alphagenome.models import variant_scorers


def score_variant(row):
    chrom, pos, ref, alt, build = row['variant_id'].split('_')

    # build variant object
    variant = genome.Variant(
        chromosome=chrom,
        position=int(pos),
        reference_bases=ref,
        alternate_bases=alt,
    )

    # 1 MB input
    interval = variant.reference_interval.resize(2**20)

    splicing_scorers = [
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_SITES'],
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_SITE_USAGE'],
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_JUNCTIONS'],
    ]
    results = {}

    scores = model.score_variant(
        interval,
        variant,
        variant_scorers=splicing_scorers,
        organism=dna_client.Organism.HOMO_SAPIENS,
    )

    df_scores = variant_scorers.tidy_scores([scores])

    return df_scores

In [14]:
def compute_merged_splicing_score(
    df: pd.DataFrame,
) -> pd.DataFrame:
  """Computes the merged splicing score from tidy variant scores.

  For each variant, takes the max raw_score across genes and tracks for each
  splicing scorer, then combines them as:
    merged_splicing = max(splice_sites) + max(splice_site_usage)
                      + max(splice_junctions) / 5

  Args:
    df: Tidy scores DataFrame from variant_scorers.tidy_scores().

  Returns:
    DataFrame with one row per variant and columns for each splicing scorer's
    max score plus the merged splicing score.
  """
  # Get the max score per variant per output type.
  df['variant_id'] = df['variant_id'].map(str)
  max_scores = (
      df.groupby(['variant_id', 'output_type'])['raw_score']
      .max()
      .reset_index()
      .pivot(index='variant_id', columns='output_type', values='raw_score')
      .fillna(0.0)
  )

  # Compute the merged splicing score with the appropriate weights.
  max_scores['alphagenome_splicing'] = (
      max_scores.get('SPLICE_SITES', 0.0)
      + max_scores.get('SPLICE_SITE_USAGE', 0.0)
      + max_scores.get('SPLICE_JUNCTIONS', 0.0) / 5.0
  )

  return max_scores.reset_index()



In [15]:
import pandas as pd


row = dfs['clinvar_noncoding'].iloc[0]
df_scores = score_variant(row)
merged_scores = compute_merged_splicing_score(df_scores)
print(merged_scores)

row

E0421 16:43:18.863581 1866190 xtile_compiler.cc:399] Fusion: gemm_fusion_dot.499 = bf16[32,2048,1024]{1,2,0} fusion(bitcast.9031, bitcast.112, bitcast.114, bitcast.116, bitcast.118), kind=kCustom, calls=gemm_fusion_dot.499_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["1","128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0421 16:43:18.863911 1866190 xtile_compiler.cc:401] Computation: gemm_fusion_dot.499_computation.clone {
  parameter_1.8 = bf16[32,128,512]{2,1,0} parameter(1)
  parameter_2.5 = bf16[32,128,512]{2,1,0} parameter(2)
  parameter_3 = bf16[32,128,512]{2,1,0} parameter(3)
  parameter_4 = bf16[32,128,512]{2,1,0} parameter(4)
  concatenate.592 = bf16[32,12

KeyboardInterrupt: 

In [ ]:
import pandas as pd


row = dfs['clinvar_noncoding'].iloc[1]
df_scores = score_variant(row)
merged_scores = compute_merged_splicing_score(df_scores)
print(merged_scores)

row

output_type        variant_id  SPLICE_JUNCTIONS  SPLICE_SITES  \
0            chr1:2520467:C>T          0.066467      0.003906   

output_type  SPLICE_SITE_USAGE  alphagenome_splicing  
0                     0.005859              0.023059  


variant_id                                       chr1_2520467_C_T_hg38
prediction                                                    0.006831
target                                                             1.0
variant_scorer                               ensemble_scorer;sj,ssu,ss
output_type          RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...
metric_calculator                      SimpleAggregatorAUPRCCalculator
metric_name                            auprc_max_abs_track_aggregation
Name: 1, dtype: object

In [ ]:
import pandas as pd


row = dfs['clinvar_noncoding'].iloc[2]
df_scores = score_variant(row)
merged_scores = compute_merged_splicing_score(df_scores)
print(merged_scores)

row

output_type        variant_id  SPLICE_JUNCTIONS  SPLICE_SITES  \
0            chr1:9943525:C>T          0.248657      0.021484   

output_type  SPLICE_SITE_USAGE  alphagenome_splicing  
0                     0.070312              0.141528  


variant_id                                       chr1_9943525_C_T_hg38
prediction                                                    0.040409
target                                                             1.0
variant_scorer                               ensemble_scorer;sj,ssu,ss
output_type          RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...
metric_calculator                      SimpleAggregatorAUPRCCalculator
metric_name                            auprc_max_abs_track_aggregation
Name: 2, dtype: object

In [1]:
cmp_df = []

for row_nr in range(10):
    row = dfs['clinvar_noncoding'].iloc[row_nr]
    df_scores = score_variant(row)
    merged_scores = compute_merged_splicing_score(df_scores)
    prediction = merged_scores['alphagenome_splicing'].values[0]
    target = row['target']
    cmp_df.append({
        'variant_id': row['variant_id'],
        'prediction': prediction,
        'target': target,
    })
    

NameError: name 'dfs' is not defined

In [15]:
from sklearn import metrics


auprc = metrics.average_precision_score(
    cmp_df['paper_target'],
    cmp_df['weighted mean']
)

print("AUPRC:", auprc)

AUPRC: 1.0


In [23]:

PREDS_PATH = 'https://storage.googleapis.com/alphagenome/evals'

evals = {
    'clinvar_splice_site_region': {
        'output_type':      'SPLICE_SITE_USAGE;RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_JUNCTIONS;SPLICE_SITES',
        'metric_name':      'auprc_max_abs_track_aggregation',
        'metric_fn':        lambda x: metrics.average_precision_score(x['target'], x['prediction']),
        'reported_metric':  0.5699,
    },
    'clinvar_noncoding': {
        'output_type':      'RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAGE;SPLICE_JUNCTIONS;SPLICE_SITES',
        'metric_name':      'auprc_max_abs_track_aggregation',
        'metric_fn':        lambda x: metrics.average_precision_score(x['target'], x['prediction']),
        'reported_metric':  0.6588,
    },
    'clinvar_missense': {
        'output_type':      'SPLICE_SITE_USAGE;SPLICE_JUNCTIONS;SPLICE_SITES;RNA_SEQ;SPLICE_SITE_POSITIONS',
        'metric_name':      'auprc_max_abs_track_aggregation',
        'metric_fn':        lambda x: metrics.average_precision_score(x['target'], x['prediction']),
        'reported_metric':  0.1792,
    }
}

for eval_name, c in evals.items():
  print(f"\nEval: {eval_name}")

  filepath = os.path.join(PREDS_PATH, eval_name + '_predictions' + '.feather')
  predictions = pd.read_feather(filepath)[1:500]

  recomputed_metric = c['metric_fn'](predictions)
  print(f"  Reported:   {c['reported_metric']}")
  print(f"  Recomputed: {recomputed_metric:.4f}")



Eval: clinvar_splice_site_region
  Reported:   0.5699
  Recomputed: 0.7883

Eval: clinvar_noncoding
  Reported:   0.6588
  Recomputed: 1.0000

Eval: clinvar_missense
  Reported:   0.1792
  Recomputed: 0.1376


In [72]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

def fit_weighted_average(cmp_df):
    sub = cmp_df.dropna(subset=['paper', 'ss', 'ssu', 'sj']).copy()

    X = sub[['ss', 'ssu', 'sj']].to_numpy()
    y = sub['paper'].to_numpy()

    model_lr = LinearRegression(fit_intercept=False)
    model_lr.fit(X, y)

    weights = model_lr.coef_
    pred = model_lr.predict(X)

    sub['weighted_pred'] = pred

    print("weights:")
    print(f"ss  = {weights[0]:.6f}")
    print(f"ssu = {weights[1]:.6f}")
    print(f"sj  = {weights[2]:.6f}")

    corr = sub['paper'].corr(sub['weighted_pred'])
    mad = (sub['paper'] - sub['weighted_pred']).abs().mean()

    print(f"\ncorrelation = {corr:.6f}")
    print(f"mean abs diff = {mad:.6f}")

    return model_lr, sub

In [73]:
model_lr, fit_df = fit_weighted_average(cmp_df)

weights:
ss  = 0.313952
ssu = 0.334135
sj  = 0.066144

correlation = 0.998202
mean abs diff = 0.019525


In [ ]:
# Debug: Compare both approaches side-by-side
def debug_compare_approaches(row):
    """Compare approach 1 (multiple calls) vs approach 2 (single call)"""
    
    # Approach 1: Multiple separate calls
    print("\n=== APPROACH 1: Multiple separate calls ===")
    res1 = score_variant_all(row)
    ss1  = res1['SPLICE_SITES']
    ssu1 = res1['SPLICE_SITE_USAGE']
    sj1  = res1['SPLICE_JUNCTIONS']
    
    weighted1 = 1 * ss1 + 1 * ssu1 + 0.2 * sj1
    print(f"ss:  {ss1:.8f}")
    print(f"ssu: {ssu1:.8f}")
    print(f"sj:  {sj1:.8f}")
    print(f"weighted: {weighted1:.8f}")
    
    # Approach 2: Single call with all scorers
    print("\n=== APPROACH 2: Single call with all scorers ===")
    chrom, pos, ref, alt, build = row['variant_id'].split('_')
    variant = genome.Variant(
        chromosome=chrom,
        position=int(pos),
        reference_bases=ref,
        alternate_bases=alt,
    )
    interval = variant.reference_interval.resize(2**20)
    
    splicing_scorers_list = [
        variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_SITES'],
        variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_SITE_USAGE'],
        variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_JUNCTIONS'],
    ]
    
    scores = model.score_variant(
        interval,
        variant,
        variant_scorers=splicing_scorers_list,
    )
    
    df_scores = variant_scorers.tidy_scores([scores])
    print("\nTidy scores shape:", df_scores.shape)
    print(df_scores[['variant_id', 'output_type', 'raw_score']].head(20))
    
    # Get max per output type
    max_per_type = df_scores.groupby('output_type')['raw_score'].max()
    print("\nMax per output type:")
    print(max_per_type)
    
    ss2  = max_per_type.get('SPLICE_SITES', 0.0)
    ssu2 = max_per_type.get('SPLICE_SITE_USAGE', 0.0)
    sj2  = max_per_type.get('SPLICE_JUNCTIONS', 0.0)
    
    weighted2 = ss2 + ssu2 + sj2 / 5.0
    print(f"\nss:  {ss2:.8f}")
    print(f"ssu: {ssu2:.8f}")
    print(f"sj:  {sj2:.8f}")
    print(f"weighted: {weighted2:.8f}")
    
    print("\n=== COMPARISON ===")
    print(f"Approach 1 weighted: {weighted1:.8f}")
    print(f"Approach 2 weighted: {weighted2:.8f}")
    print(f"Difference: {abs(weighted1 - weighted2):.8f}")
    print(f"Relative diff: {abs(weighted1 - weighted2) / max(abs(weighted1), abs(weighted2)) * 100:.2f}%")
    
    return {'approach1': weighted1, 'approach2': weighted2}

# Test on first variant
test_row = dfs['clinvar_noncoding'].iloc[0]
debug_result = debug_compare_approaches(test_row)


In [ ]:
# this is prediction using manuelly calcualted scores: ref - alt
# I should probably this the score variant function than this

'''
all_ontologies = metadata.splice_site_usage['ontology_curie'].unique().tolist() # because it doesn't specify one ontology


def predict_row(row):
    # Parse variant
    chrom, pos, ref, alt, build = row['variant_id'].split('_')
    variant = genome.Variant(
        chromosome=chrom,
        position=int(pos),
        reference_bases=ref,
        alternate_bases=alt,
    )
    interval = variant.reference_interval.resize(2**20)
    print(variant)
    print(interval)

    # Predict
    result = model.predict_variant(
        interval,
        variant,
        requested_outputs=[
            dna_model.OutputType.SPLICE_SITES,
            dna_model.OutputType.SPLICE_SITE_USAGE,
            dna_model.OutputType.SPLICE_JUNCTIONS,
        ],
        ontology_terms=all_ontologies,
    )

    # Compute diff for each output type
    diff_ss  = result.alternate.splice_sites.values - result.reference.splice_sites.values
    diff_ssu = result.alternate.splice_site_usage.values - result.reference.splice_site_usage.values
    diff_sj  = result.alternate.splice_junctions.values - result.reference.splice_junctions.values

    max_ss  = float(np.max(np.abs(diff_ss)))
    max_ssu = float(np.max(np.abs(diff_ssu)))
    max_sj  = float(np.max(np.abs(diff_sj)))

    print(f"max abs SPLICE_SITES     : {max_ss:.6f}")
    print(f"max abs SPLICE_SITE_USAGE: {max_ssu:.6f}")
    print(f"max abs SPLICE_JUNCTIONS : {max_sj:.6f}")

    # Aggregation
    my_score = float(np.max(np.abs(np.concatenate([
        diff_ss.flatten(),
        diff_ssu.flatten(),
        diff_sj.flatten(),
    ]))))

    print(f"\nPaper prediction : {row['prediction']:.6f}")
    print(f"My prediction    : {my_score:.6f}")
    return my_score

# Test on first row
row = dfs['clinvar_noncoding'].iloc[1]
print(row)
score = predict_row(row)
'''